# 10 · Clases desbalanceadas: pesos, sampling y métricas correctas

Fraude, fallas, abandono y eventos clínicos/operacionales suelen tener pocos positivos. El objetivo no es 'balancear por balancear', sino tomar mejores decisiones con eventos raros.

## Objetivos
- Entender por qué accuracy falla con rare events.
- Comparar class weights, undersampling y oversampling.
- Usar SMOTE correctamente **dentro de CV**.
- Entender cuándo el oversampling empeora calibración.
- Evaluar con PR-AUC, recall, precision@K y costos.


In [ ]:
!pip -q install imbalanced-learn
import numpy as np, pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, average_precision_score, roc_auc_score, confusion_matrix
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler
SEED=42
X,y=make_classification(n_samples=12000,n_features=30,n_informative=10,n_redundant=8,weights=[.985,.015],class_sep=1.2,flip_y=.003,random_state=SEED)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.3,stratify=y,random_state=SEED)
print(pd.Series(ytr).value_counts(), 'prevalencia=',ytr.mean())

## 1. Baseline sin tratamiento
Un clasificador que siempre dice 0 logra ~98.5% accuracy. Por eso miramos ranking y desempeño sobre el positivo.


In [ ]:
def report(name,m):
 m.fit(Xtr,ytr); p=m.predict_proba(Xte)[:,1]; pred=(p>=.5).astype(int)
 print('\n',name,'ROC=',round(roc_auc_score(yte,p),3),'PR=',round(average_precision_score(yte,p),3)); print(classification_report(yte,pred,digits=3))
report('Logistic',LogisticRegression(max_iter=5000))
report('Logistic balanced',LogisticRegression(max_iter=5000,class_weight='balanced'))

## 2. Class weights
Cambian el costo de equivocarse durante entrenamiento sin duplicar filas. Suelen ser un primer enfoque fuerte. Pero el threshold 0.5 deja de tener el mismo significado probabilístico y puede requerirse calibración posterior.


## 3. Random under/over sampling
- Undersampling reduce negativos: más rápido, pero elimina información.
- Oversampling duplica positivos: conserva negativos, pero puede aumentar overfitting.
- SMOTE genera puntos sintéticos interpolando vecinos positivos. Funciona mejor en espacios continuos razonables; no es mágico y puede crear ejemplos poco plausibles.

**Crítico:** resamplear antes del split es leakage. Sampling debe ocurrir únicamente dentro del training fold.


In [ ]:
cv=StratifiedKFold(5,shuffle=True,random_state=SEED)
methods={
 'none':ImbPipeline([('model',LogisticRegression(max_iter=5000))]),
 'weight':ImbPipeline([('model',LogisticRegression(max_iter=5000,class_weight='balanced'))]),
 'over':ImbPipeline([('sample',RandomOverSampler(random_state=SEED)),('model',LogisticRegression(max_iter=5000))]),
 'under':ImbPipeline([('sample',RandomUnderSampler(random_state=SEED)),('model',LogisticRegression(max_iter=5000))]),
 'smote':ImbPipeline([('sample',SMOTE(random_state=SEED,k_neighbors=5)),('model',LogisticRegression(max_iter=5000))])
}
rows=[]
for name,m in methods.items():
 s=cross_validate(m,Xtr,ytr,cv=cv,scoring=['average_precision','roc_auc','recall','precision'],n_jobs=-1)
 rows.append([name,*[s['test_'+k].mean() for k in ['average_precision','roc_auc','recall','precision']]])
pd.DataFrame(rows,columns=['metodo','PR-AUC','ROC-AUC','recall@.5','precision@.5']).round(3)

## 4. Modelos de árboles y imbalance
Random Forest soporta `class_weight`. XGBoost usa `scale_pos_weight`. LightGBM/CatBoost tienen opciones equivalentes. Ajustar pesos puede cambiar ranking y distribución de scores; valida con PR-AUC y la métrica operacional.


In [ ]:
for name,m in [
 ('RF',RandomForestClassifier(n_estimators=400,min_samples_leaf=2,random_state=SEED,n_jobs=-1)),
 ('RF balanced',RandomForestClassifier(n_estimators=400,min_samples_leaf=2,class_weight='balanced_subsample',random_state=SEED,n_jobs=-1))
]: report(name,m)

## 5. Imbalance extremo: piensa en ranking y recuperación
Con 0.01% de positivos, un threshold global puede ser menos útil que top-K, alert budget, anomaly detection, positive-unlabeled learning o cascadas de modelos. Si labels positivos son incompletos, el problema no es solo imbalance: es **label quality**.

## Alternativas avanzadas
- focal loss en deep learning;
- Balanced Random Forest;
- EasyEnsemble/RUSBoost;
- hard-negative mining;
- cost-sensitive learning;
- PU learning;
- one-class/anomaly methods si casi no hay positivos.

## Errores comunes
- SMOTE antes del split;
- sintetizar variables categóricas como si fueran continuas;
- reportar accuracy;
- cambiar prevalence artificialmente y creer que probabilidades siguen calibradas;
- optimizar recall al 100% sin considerar carga de falsos positivos.

## Ejercicios
1. Compara `SMOTENC` con SMOTE en datos categóricos.
2. Evalúa precision@100 y recall@100.
3. Calibra el modelo luego de class weighting.
4. Busca un threshold que respete máximo 200 alertas.
5. Simula prevalence 1%, 0.1%, 0.01% y analiza PR-AUC.
6. Implementa focal loss en PyTorch.
